## **MLlib 튜토리얼**

In [6]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local').appName('mlib-tutorial').getOrCreate()

### **MLlib 예제: Logistic Regression**
+ ```pyspark.ml.linalg```: 선형대수(linear algebra) 관련 패키지
    + ```Vectors```: 벡터 데이터 타입
+ ```pyspark.ml.classification```: 분류 모델 관련 패키지
    + ```LogisticRegression```: 가장 기본적인 분류 모델
+ ```딥러닝보다 머신러닝

In [3]:
# Vectors, LogisticRegression 임포트
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import LogisticRegression

**데이터 정의: 레이블과 특징**
+ ```dense()```: 모든 값을 저장하는 벡터 생성
+ ```sparse()```: 0이 아닌 값만 저장하는 벡터 생성

In [8]:
# 훈련 데이터: (label, features)
train_data = [
    (1.0, Vectors.dense([0.0, 1.1, 0.1])),
    (0.0, Vectors.dense([2.0, 1.0, -1.0])),
    (0.0, Vectors.dense([2.0, 1.3, 1.0])),
    (1.0, Vectors.dense([0.0, 1.2, -0.5])),
]

In [4]:
# [+] 스키마 정의
schema = ['label','features']

In [9]:
# [+] 데이터프레임 생성
train_df = spark.createDataFrame(train_data, schema)

In [10]:
# 데이터프레임 출력 
train_df.show()

+-----+--------------+
|label|      features|
+-----+--------------+
|  1.0| [0.0,1.1,0.1]|
|  0.0|[2.0,1.0,-1.0]|
|  0.0| [2.0,1.3,1.0]|
|  1.0|[0.0,1.2,-0.5]|
+-----+--------------+



**LogisticRegression (Estimator) 생성**
+ ```maxIter```: 최대 학습 횟수 매개변수
+ ```regParam```: 정규화 매개변수

In [11]:
# [+] Estimator(학습 알고리즘) 객체 생성
lr = LogisticRegression(maxIter = 30, regParam = 0.01) 

In [13]:
# [+] 모델 학습
model = lr.fit(train_df)

26/11/24 04:24:25 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/24 04:24:25 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [14]:
# 모델 출력
model

LogisticRegressionModel: uid=LogisticRegression_d11a54e6e1b9, numClasses=2, numFeatures=3

In [15]:
# 테스트셋
test_data = [
    (1.0, Vectors.dense([-1.0, 1.5, 1.3])),
    (0.0, Vectors.dense([3.0, 2.0, -0.1])),
    (1.0, Vectors.dense([0.0, 2.2, -1.5])),
]

In [17]:
# [+] 데이터프레임 생성
test_df = spark.createDataFrame(test_data, schema)

In [19]:
# [+] 모델 예측
predictions = model.transform(test_df)

In [20]:
# 예측결과 출력
predictions.show()

+-----+--------------+--------------------+--------------------+----------+
|label|      features|       rawPrediction|         probability|prediction|
+-----+--------------+--------------------+--------------------+----------+
|  1.0|[-1.0,1.5,1.3]|[-6.2435550918400...|[0.00193916823498...|       1.0|
|  0.0|[3.0,2.0,-0.1]|[5.45228608726759...|[0.99573180142693...|       0.0|
|  1.0|[0.0,2.2,-1.5]|[-4.4104172202339...|[0.01200425500655...|       1.0|
+-----+--------------+--------------------+--------------------+----------+



### **LogisticRegression 파이프라인 구현하기**
+ 예측 문제: Spark에 대한 텍스트 인지 아닌지를 분류
+ ```Pipeline```: 머신러닝 과정에 대한 파이프라인 정의
+ ```HashingTF```: 컬럼을 용어빈도(term-frequency) 컬럼으로 변환하는 Transformer
+ ```Tokenizer```: 컬럼의 텍스트를 여러 개의 단어로 분할하는 Transformer

In [21]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, Tokenizer

In [22]:
# 훈련 데이터를 데이터프레임으로 생성
train_df = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])

In [23]:
train_df.show()

+---+----------------+-----+
| id|            text|label|
+---+----------------+-----+
|  0| a b c d e spark|  1.0|
|  1|             b d|  0.0|
|  2|     spark f g h|  1.0|
|  3|hadoop mapreduce|  0.0|
+---+----------------+-----+



**머신러닝 수행 과정**

Tokenizer ${\rightarrow}$ HashingTF ${\rightarrow}$ LogisticRegression

In [25]:
# [+] Tokenizer 객체 생성
tokenizer = Tokenizer(inputCol = 'text', outputCol = 'words')

In [26]:
# 토크나이징 수행
df_tok = tokenizer.transform(train_df)
df_tok.show()

+---+----------------+-----+--------------------+
| id|            text|label|               words|
+---+----------------+-----+--------------------+
|  0| a b c d e spark|  1.0|[a, b, c, d, e, s...|
|  1|             b d|  0.0|              [b, d]|
|  2|     spark f g h|  1.0|    [spark, f, g, h]|
|  3|hadoop mapreduce|  0.0| [hadoop, mapreduce]|
+---+----------------+-----+--------------------+



In [27]:
# [+] HashingTF 객체 생성
#실제 파이프라인에 아직 word 생성되지 않았다
hashingTF = HashingTF(inputCol = tokenizer.getOutputCol(), outputCol = 'features')

In [28]:
# 단어빈도 벡터로 변환
df_tf = hashingTF.transform(df_tok)
df_tf.show()

+---+----------------+-----+--------------------+--------------------+
| id|            text|label|               words|            features|
+---+----------------+-----+--------------------+--------------------+
|  0| a b c d e spark|  1.0|[a, b, c, d, e, s...|(262144,[74920,89...|
|  1|             b d|  0.0|              [b, d]|(262144,[89530,14...|
|  2|     spark f g h|  1.0|    [spark, f, g, h]|(262144,[36803,17...|
|  3|hadoop mapreduce|  0.0| [hadoop, mapreduce]|(262144,[132966,1...|
+---+----------------+-----+--------------------+--------------------+



In [33]:
df_tf.select('features').show()

+--------------------+
|            features|
+--------------------+
|(262144,[74920,89...|
|(262144,[89530,14...|
|(262144,[36803,17...|
|(262144,[132966,1...|
+--------------------+



**LogisticRegression 객체 생성**
+ ```maxIter```: 10
+ ```regParam```: 0.001

In [30]:
# [+] Estimator 객체 생성
lr = LogisticRegression(maxIter = 10, regParam = 0.001)

In [31]:
# 모델 학습
model = lr.fit(df_tf)

In [32]:
# 모델 학습 결과(특징별 계수인터셉트)
print(model.coefficients)
print(model.intercept)

(262144,[36803,74920,89530,107107,132966,148981,167694,173558,198017,209078,228158],[2.214079525762639,2.6396709316511835,-0.725682486825636,2.6396709316511835,-1.2465028766617914,-0.725682486825636,2.6396709316511835,3.6403128430603697,-1.2465028766617914,2.214079525762639,2.214079525762639])
-4.169141395340048


**머신러닝 파이프라인 생성 및 실행**\
```Pipeline.fit()```은 다음의 절차들을 일괄적으로 수행
1. 토크나이징(Tokenizer): text ${\rightarrow}$ words
2. 단어빈도(Term-Frequency) 벡터로 변환: words ${\rightarrow}$ features
3. 로지스틱스 회귀모델(LogisticRegression) 학습

In [34]:
# [+] Pipeline 객체 생성
pipeline = Pipeline(stages = [tokenizer, hashingTF, lr])

In [36]:
# [+] Pipeline 실행
model = pipeline.fit(train_df)

In [37]:
# 테스트 데이터에 대한 DataFrame 생성
test_df = spark.createDataFrame([
    (4, "spark i j k"),
    (5, "l m n"),
    (6, "spark hadoop spark"),
    (7, "apache hadoop")
], ["id", "text"])

In [39]:
# [+] 테스트 데이터에 대한 예측
predictions = model.transform(test_df)

In [40]:
predictions.show()

+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
| id|              text|               words|            features|       rawPrediction|         probability|prediction|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
|  4|       spark i j k|    [spark, i, j, k]|(262144,[19036,68...|[0.52882855227967...|[0.62920984896684...|       0.0|
|  5|             l m n|           [l, m, n]|(262144,[1303,526...|[4.16914139534004...|[0.98477000676230...|       0.0|
|  6|spark hadoop spark|[spark, hadoop, s...|(262144,[173558,1...|[-1.8649814141189...|[0.13412348342566...|       1.0|
|  7|     apache hadoop|    [apache, hadoop]|(262144,[68303,19...|[5.41564427200183...|[0.99557321143985...|       0.0|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+

